# Imports

In [1]:
# ML
import pandas as pd
import numpy as np
import pyarrow  # ensures pandas can read/write parquet
import math

# Models
from bertopic import BERTopic
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# System
import gc
from pathlib import Path
from datetime import datetime
import subprocess

# Pyton
from functools import wraps
from typing import Iterator, List, Tuple

# Visualization
import matplotlib.pyplot as plt
from IPython.display import display

# Loading and Exporting


In [2]:
dataset_name = "Labeled_Datasets/Ecuador"  # folder path
raw_file_name = "Ecuador_part_all.gzip.parquet"  # file name
date = "2026_01_10"

In [3]:
raw_file_name_no_suffixes = raw_file_name.split(".")[0]
processed_file_name = f"Topic_Ner_{raw_file_name}"

folder_path = Path("../results") / dataset_name / raw_file_name_no_suffixes / date
processed_df_path = folder_path / processed_file_name

print(f"folder_path: {folder_path}")
print(f"processed_df_path: {processed_df_path}")

folder_path: ../results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10
processed_df_path: ../results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10/Topic_Ner_Ecuador_part_all.gzip.parquet


In [4]:
df = pd.read_parquet(processed_df_path)

# Utils

## Logs

In [5]:
REPORT_PATH = folder_path / "progress_report.txt"

def write_report(msg: str):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} | {msg}"

    print(line)
    with open(REPORT_PATH, "a") as f:
        f.write(line + "\n")


def report_done(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            write_report(f"Completed: {func.__name__}")
            return result
        except Exception as e:
            write_report(f"FAILED: {func.__name__} | {e}")
            raise
    return wrapper

## Save df

In [6]:
def save_df(
        df: pd.DataFrame,
        folder_path: str | Path = folder_path,
        file_name: str = raw_file_name
    )-> None :
    df_output_path = folder_path / f"processed_{file_name}"
    df.to_parquet(df_output_path, index=False, compression="gzip")
    write_report(f"df is saved to: {df_output_path}")

## Loading Report

In [7]:
write_report(f"Loaded sd df: {processed_file_name}.")
write_report(f"df shape: {df.shape}")
print("df head:")
display(df.head(3))

2026-01-14 15:27:55 | Loaded sd df: Topic_Ner_Ecuador_part_all.gzip.parquet.
2026-01-14 15:27:55 | df shape: (1468565, 22)
df head:


,postid,post_text,application_name,post_language,in_reply_to_postid,in_reply_to_accountid,post_time,accountid,account_profile_description,follower_count,...,is_repost,reposted_accountid,reposted_postid,hashtags,urls,account_mentions,is_control,clean_text_with_entities,clean_text_without_enteties,translated_post
0,0bafe020a260495bee38b98fde83668d924454019b,Vamos a cambiar la forma de hacer comunicación...,2727201c363b6fc4afe259ba4094ae0c49cb4df224,spa_Latn,None,None,2010-03-13 02:52:00,22de67d991b31493165b9270f7d8b5b4ce712d5fc8,"#Periodismo, #RelacionesPúblicas, #Comunicac...",1528,...,False,None,None,[],[],[],False,vamos cambiar la forma de hacer comunicación c...,vamos cambiar la forma de hacer comunicación c...,going change way corporate communication done.
1,608fba51f2460c7f1ca527dfc48de2bc858d4ad201,"Damos servicio de manejo de prensa, elaboració...",2727201c363b6fc4afe259ba4094ae0c49cb4df224,spa_Latn,None,None,2010-03-13 03:08:00,22de67d991b31493165b9270f7d8b5b4ce712d5fc8,"#Periodismo, #RelacionesPúblicas, #Comunicac...",1528,...,False,None,None,[],[],[],False,"damos servicio de manejo de prensa, elaboració...","damos servicio de manejo de prensa, elaboració...","provide media management services, communicati..."
2,514b2c38b26e240cac85382be74ed7d61a02468ac1,@db7c47cef2aa2ba68b5c010d761a03fc1c0f0d1302 Bu...,2727201c363b6fc4afe259ba4094ae0c49cb4df224,spa_Latn,f29b32f8dbaa1e0f00152decbcc2bd93dddc740464,db7c47cef2aa2ba68b5c010d761a03fc1c0f0d1302,2010-03-13 15:26:00,2c6c0ef5f445760177b016e33b36809bde3c7ac241,None,35,...,False,None,None,[],[],[db7c47cef2aa2ba68b5c010d761a03fc1c0f0d1302],False,mention_db7c47cef2aa2ba68b5c010d761a03fc1c0f0d...,"<user> bueno se estan destapando mascaras, ya ...","well, masks unveiled, new movie ""the reloader ..."


# English Topic Clustering

Note: the topic clustering doent influenced by the entities since it is preformed on the text without the entitites.

## Utils

### Posts Topic Stats

In [8]:
@report_done
def make_topic_stats(df, topic_col_name):
    """
    Calculate per-topic post counts.

    Args:
        df (pd.DataFrame): Data with topics and 'is_control'.
        topic_col_name (str): Column name of topic labels.

    Returns:
        topic_stats (pd.DataFrame): Colums: [topic, total_count, io_Count, legitimate_posts, io_percent].
    """
    topic_stats = (
        df.groupby(topic_col_name)
          .agg(
              total_count=(topic_col_name, "count"),
              io_count=("is_control", lambda x: (~x).sum())
          )
          .reset_index()
    )
    topic_stats["legitimate_posts"] = topic_stats["total_count"] - topic_stats["io_count"]
    topic_stats["io_percent"] = topic_stats["io_count"] / topic_stats["total_count"] * 100
    return topic_stats

In [9]:
from matplotlib.lines import Line2D

def plot_topic_stats(topic_stats, topic_col_name, model_name, output_folder_path: str | Path = ""):
    """
    Plot stacked IO vs legitimate post counts per topic.
    Saves the fig to output_path if provided.

    Args:
        topic_stats (pd.DataFrame): Output of make_topic_stats().
        topic_col_name (str): Topic column to plot.
        output_folder_path (str): Path to *folder* save the plot.

    Returns:
        None
    """
    topics = topic_stats[topic_col_name]
    io_count = topic_stats["io_count"]
    legitimate_count = topic_stats["legitimate_posts"]
    io_percent = topic_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO posts
    bar_io = plt.bar(topics, io_count, label="IO posts", color="red")

    # 🔵 Top: Non-IO posts stacked above IO
    bar_legit = plt.bar(topics, legitimate_count, bottom=io_count, label="Non-IO posts")

    # Add IO percentage text
    ax = plt.gca()
    for x, io, legit, pct in zip(topics, io_count, legitimate_count, io_percent):
        total_height = io + legit
        ax.text(
            x,
            total_height + 0.5,              # slightly above the bar
            f"{pct:.1f}%",                 # format like 23.5%
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular"
        )

    plt.title(f"Post Count per Topic ({model_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Posts")
    plt.xticks(topics)
    plt.grid(axis="y")

    percent_legend = Line2D([], [], color="none")
    total_posts_legend = Line2D([], [], color="none")
    plt.legend(
        handles=[bar_io, bar_legit, percent_legend, total_posts_legend],
        labels=[
            "IO posts",
            "Non-IO posts",
            "% IO Posts Ratio",
            f"Total posts: {topic_stats['total_count'].sum()}",
        ],
        loc="best",
    )


    plt.tight_layout()


    if output_folder_path:
        output_folder_path = Path(output_folder_path)
        plot_file_path = output_folder_path / (f"{model_name}_topic_clustering.png")
        plt.savefig(plot_file_path, dpi=300)
        print(f"plot is saved to {plot_file_path}")

    plt.show()

    plt.close()

### Accoutns Topic Stats

In [10]:
@report_done
def make_topic_account_stats(
    df: pd.DataFrame,
    topic_col: str,
    author_col: str = "accountid",
    is_control_col: str = "is_control",
    threshold: float = 0.5,
) -> pd.DataFrame:
    """Compute per-topic unique account counts (IO vs control) using an author-level threshold.

    Args:
        df: Post-level DataFrame.
        topic_col: Topic column name.
        author_col: Author identifier column.
        is_control_col: Column where 1 = control, 0 = IO.
        threshold: IO-rate threshold to label an author as IO (1 = IO).

    Returns:
        DataFrame with per-topic counts of unique IO/control accounts and IO percentage.
    """
    # Author-level IO rate: 1 - mean(is_control)
    author_io_rate = 1 - df.groupby(author_col)[is_control_col].mean()
    author_io_tag = (author_io_rate >= threshold).astype(int)

    topic_author_df = df[[topic_col, author_col]].drop_duplicates().copy()
    topic_author_df["author_io_tag"] = topic_author_df[author_col].map(author_io_tag)

    stats = (
        topic_author_df
        .groupby(topic_col)["author_io_tag"]
        .value_counts()
        .unstack(fill_value=0)
    )

    # Ensure both columns exist even if absent in data
    stats = stats.reindex(columns=[0, 1], fill_value=0).rename(
        columns={0: "control_accounts", 1: "io_accounts"}
    ).reset_index()

    stats["total_accounts"] = stats["control_accounts"] + stats["io_accounts"]
    stats["io_percent"] = (stats["io_accounts"] / stats["total_accounts"]).mul(100)

    return stats

In [11]:
def plot_topic_account_stats(
    topic_account_stats: pd.DataFrame,
    topic_col_name: str,
    model_name: str,
    output_folder_path: Path | str | None = None,
):
    """Plot IO vs control account counts per topic.

    Args:
        topic_account_stats: Output of make_topic_account_stats().
        topic_col_name: Topic column name.
        model_name: Model name for title / filename.
        output_folder_path: Optional folder to save the plot.

    Returns:
        None
    """
    topics = topic_account_stats[topic_col_name]
    io_count = topic_account_stats["io_accounts"]
    control_count = topic_account_stats["control_accounts"]
    io_percent = topic_account_stats["io_percent"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO accounts
    bar_io = plt.bar(topics, io_count, label="IO accounts", color="red")

    # 🔵 Top: Control accounts
    bar_control = plt.bar(
        topics,
        control_count,
        bottom=io_count,
        label="Control accounts",
    )

    # Add IO percentage text
    ax = plt.gca()
    for x, io, ctrl, pct in zip(topics, io_count, control_count, io_percent):
        total = io + ctrl
        ax.text(
            x,
            total + 0.3,
            f"{pct:.1f}%",
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="regular",
        )

    plt.title(f"Unique Accounts per Topic ({model_name})")
    plt.xlabel("Topic ID")
    plt.ylabel("Number of Unique Accounts")
    plt.xticks(topics)
    plt.grid(axis="y")

    # Legend with extra info lines
    percent_legend = Line2D([], [], color="none")
    total_accounts_legend = Line2D([], [], color="none")

    plt.legend(
        handles=[bar_io, bar_control, percent_legend, total_accounts_legend],
        labels=[
            "IO accounts",
            "Control accounts",
            "% IO Accounts Ratio",
            f"Total accounts: {topic_account_stats["total_accounts"].sum()}",
        ],
        loc="best",
    )

    if output_folder_path:
        output_folder_path = Path(output_folder_path)
        plot_file_path = output_folder_path / f"{model_name}_topic_accounts.png"
        plt.savefig(plot_file_path, dpi=300)
        print(f"Plot saved to {plot_file_path}")

    plt.show()
    plt.close()

## Embeddings

In [12]:
from sentence_transformers import SentenceTransformer

post_text_col = "translated_post"

posts_texts = df[post_text_col].astype(str).tolist()

model = SentenceTransformer("digio/Twitter4SSE")

embeddings = model.encode(
    posts_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True, 
)

embeddings = embeddings.astype("float32", copy=False) # in order to use less memory
write_report(f"Generated embeddings shape: {embeddings.shape}")

No sentence-transformers model found with name digio/Twitter4SSE. Creating a new one with mean pooling.


Batches:   0%|          | 0/45893 [00:00<?, ?it/s]

2026-01-14 15:37:35 | Generated embeddings shape: (1468565, 768)


## BERTopic

[BERTopic Parametes](https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html#calculate_probabilities)

In [13]:
min_topic_size = int(min(len(df) / 10, int(math.log(100, 1.1)))) # 100 -> 10, 50K -> 113, 1M -> 144
chunk_size = 2**17 # 131,072
BERTopic_topics_col_name = "BERTopic_topic"
BERTopic_probs_col_name = "BERTopic_prob"

### Build Chunks Models

https://maartengr.github.io/BERTopic/getting_started/merge/merge.html

In [14]:
# Make output folder for chunk models
chunk_models_output_folder_path = Path(folder_path) / "BERTopic_models"
chunk_models_output_folder_path.mkdir(exist_ok=True)

In [15]:
def iter_text_chunks(
    texts: List[str],
    chunk_size: int,
) -> Iterator[Tuple[int, List[str]]]:
    """Yield (start_index, text_chunk) pairs of fixed size.

    Args:
        texts: List of documents.
        chunk_size: Max number of documents per chunk.

    Yields:
        Tuple[int, List[str]]: (start_index, chunk_texts).
    """
    for start in range(0, len(texts), chunk_size):
        yield start, texts[start : start + chunk_size]


@report_done
def build_and_save_chunk_models(
    texts: List[str],
    embeddings: np.ndarray,
    chunk_size: int,
    min_topic_size: int,
    output_dir: str | Path,
) -> None:
    """Train BERTopic models on chunks and save each to disk.

    Args:
        texts: Documents.
        embeddings: Precomputed embeddings (n_docs, dim).
        chunk_size: Max docs per chunk.
        min_topic_size: BERTopic min_topic_size.
        output_dir: Directory to save chunk models.
    """
    
    for idx, (start, text_chunk) in enumerate(iter_text_chunks(texts, chunk_size)):
        end = start + len(text_chunk)
        emb_chunk = embeddings[start:end]

        model = BERTopic(
            language="english",
            min_topic_size=min_topic_size,
            low_memory=True,
            calculate_probabilities=False,
        )

        model.fit(text_chunk, emb_chunk)

        model_path = output_dir / f"chunk_model_{idx}"
        model.save(model_path)
        write_report(f"Saved chunk model {idx} for texts {start} to {end} at {model_path}")

        # free RAM
        del model, emb_chunk, text_chunk # delete references
        gc.collect() # Garbage Collector

### Merge Chunks Models

In [ ]:
@report_done
def load_and_merge_BERTopic_models(
    chunk_models_dir: str | Path,
    min_similarity: float = 0.7, # default value is 0.7 in BERTopic docs
    folder_path: str | Path | None= None,
) -> BERTopic:
  
    chunk_models_dir = Path(chunk_models_dir)
    paths = [p for p in chunk_models_dir.iterdir()]

    if not paths:
        raise ValueError(f"No chunk models found in {chunk_models_dir}")
    
    merged_model  = None

    for p in paths:
        chunk_model = BERTopic.load(p)

        if merged_model is None: # first model
            merged_model = chunk_model
        else:
            merged_model = BERTopic.merge_models([merged_model, chunk_model], min_similarity=min_similarity)
            del chunk_model
            gc.collect()


    if merged_model is None:
        raise RuntimeError("merged_model is None after loading chunk models.")
    

    if folder_path is not None:
        merged_model.save(folder_path / "merged_BERTopic_model")
        write_report(f"Merged model saved to {folder_path / 'merged_BERTopic_model'}")
    else:
        merged_model.save(chunk_models_dir / "merged_BERTopic_model")
        write_report(f"Merged model saved to {chunk_models_dir / 'merged_BERTopic_model'}")

    return merged_model

### Run BERTopic Modeling

In [ ]:
build_and_save_chunk_models(
    texts=posts_texts,
    embeddings=embeddings,
    chunk_size=chunk_size,
    min_topic_size=min_topic_size,
    output_dir=chunk_models_output_folder_path,
)

merge_BERTopic = load_and_merge_BERTopic_models(chunk_models_dir = chunk_models_output_folder_path)
BERTopic_topic, BERT_probs = merge_BERTopic.transform(posts_texts, embeddings)

df[BERTopic_topics_col_name] = BERTopic_topic
df[BERTopic_probs_col_name] = BERT_probs

2026-01-14 15:40:01,403 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


2026-01-14 15:40:17 | Saved chunk model 0 for texts 0 to 131072 at ../results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10/BERTopic_models/chunk_model_0


2026-01-14 15:43:50,042 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


2026-01-14 15:43:57 | Saved chunk model 1 for texts 131072 to 262144 at ../results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10/BERTopic_models/chunk_model_1


2026-01-14 15:45:24,420 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


...

2026-01-14 15:53:46 | Saved chunk model 3 for texts 393216 to 524288 at ../results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10/BERTopic_models/chunk_model_3


2026-01-14 15:56:39,515 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


2026-01-14 15:56:47 | Saved chunk model 4 for texts 524288 to 655360 at ../results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10/BERTopic_models/chunk_model_4


2026-01-14 15:59:43,096 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


2026-01-14 15:59:52 | Saved chunk model 5 for texts 655360 to 786432 at ../results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10/BERTopic_models/chunk_model_5


### Save Model and df

In [ ]:
merge_BERTopic.save(folder_path / "Final_BERTopic_model")

save_df(df)
write_report("Saved df with BERTopic topics.")

### Graph and Statstistics

In [ ]:
try:
    fig = merge_BERTopic.visualize_topics(custom_labels=True)
    display(fig)   # show it
except Exception as e:
    print("Not enough topics for visualization")
    print("Error:", e)

In [ ]:
merge_BERTopic.get_topic_info()

Topic Post Statstistics

In [ ]:
BERT_topic_stats = make_topic_stats(df, BERTopic_topics_col_name)
plot_topic_stats(BERT_topic_stats, BERTopic_topics_col_name, "BERTopic", folder_path)

Topic Account Statstistics

In [ ]:
topic_account_stats = make_topic_account_stats(df, BERTopic_topics_col_name)
display(topic_account_stats)
plot_topic_account_stats(topic_account_stats, BERTopic_topics_col_name, "BERTopic", folder_path)

## K-Means

In [ ]:
kmeans_topic_col_name = "K_Means_topic"

### Run K-Means

In [ ]:
@report_done
def auto_kmeans_topics(embeddings, k_min=2**2, k_max=2**6, step = 4):
    best_k = None
    best_score = -1
    best_labels = None

    # print memory summary
    write_report(f"auto_kmeans_topics: \n {subprocess.check_output('nvidia-smi', shell=True).decode()}")

    for k in range(k_min, k_max + 1, step):
        print(f"Trying k={k}")
        km = KMeans(n_clusters=k, random_state=42, n_init="auto")
        labels = km.fit_predict(embeddings)
        score = silhouette_score(embeddings, labels)

        if score > best_score:
            best_score = score
            best_k = k
            best_labels = labels

    write_report(f"Best k for K-Means: {best_k} with silhouette score: {best_score:.4f}")

    return best_labels, best_k

In [ ]:
labels, best_k = auto_kmeans_topics(embeddings, k_min=4, k_max=50)
df[kmeans_topic_col_name] = labels
df.head()

### Graph and Statstistics

Topic Post Statstistics

In [ ]:
topic_stats = make_topic_stats(df, kmeans_topic_col_name)
plot_topic_stats(topic_stats, kmeans_topic_col_name, "K-Means", folder_path)

Topic Account Statstistics

In [ ]:
topic_account_stats = make_topic_account_stats(df, kmeans_topic_col_name)
plot_topic_account_stats(topic_account_stats, kmeans_topic_col_name, "K_Means", folder_path)

### Save Model and df

In [ ]:
save_df(df)
write_report("Saved df with BERTopic topics.")